# Modelo de Lenguaje GRU pequeño

El siguiente notebook contiene cómo: cargar textos, tokenizar, construir ventanas de entrenamiento, entrenar un modelo `Embedding -> GRU -> Linear`, medir perplexity y generar muestras.

Este notebook queda configurado por defecto para `multi3_nlu_es_hotels`, el dominio de hoteles de Multi3-NLU++. Si quieres volver a una prueba ultrarrapida, cambia `DATASET_NAME` a `toy_sintetico`.


## 1. Instalacion y entorno

En Colab puedes activar GPU en `Runtime > Change runtime type > T4 GPU` o similar. El codigo tambien funciona en CPU para los corpus pequenos, solo tarda mas.


In [61]:
#@title Instalacion y entorno
!pip -q install datasets sentencepiece "sympy>=1.13.3,<1.14"

import csv
import glob
import json
import math
import random
import re
import subprocess
from collections import Counter
from pathlib import Path

# Workaround for occasional Colab/PyTorch/SymPy import-state mismatches.
# If this cell was already run before the install line, restart the runtime once.
import sympy.core.symbol

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if device == 'cuda':
    print('gpu:', torch.cuda.get_device_name(0))


device: cpu


## 2. Configuracion

Para reproducir literalmente la version minima del reporte, deja `TOKENIZER = 'word'` o usa `char` si quieres un corpus ultrapequeno. El modelo por defecto usa embeddings de 128 y GRU de 256, dentro del rango recomendado.


In [62]:
#@title Hiperparametros y dataset
DATASET_NAME = 'multi3_nlu_es_hotels' #@param ['multi3_nlu_es_hotels', 'multi3_nlu_es_banking', 'multi3_nlu_es_all', 'toy_sintetico', 'simple_tico19_es', 'massive_es', 'mtop_es', 'tico19_es']
TOKENIZER = 'word' #@param ['word', 'char']
TEXT_FIELD_SIMPLE_TICO = 'simplification' #@param ['simplification', 'original']

LOWERCASE = True #@param {type:'boolean'}
NORMALIZE_SLOTS = True #@param {type:'boolean'}
MAX_TEXTS = 20000 #@param {type:'integer'}
MIN_FREQ = 1 #@param {type:'integer'}

SEQ_LEN = 48 #@param {type:'integer'}
STRIDE = 8 #@param {type:'integer'}
BATCH_SIZE = 64 #@param {type:'integer'}
EPOCHS = 11 #@param {type:'integer'}

EMB_DIM = 128 #@param {type:'integer'}
HIDDEN_DIM = 256 #@param {type:'integer'}
NUM_LAYERS = 1 #@param {type:'integer'}
LR = 1e-3 #@param {type:'number'}
WEIGHT_DECAY = 1e-4 #@param {type:'number'}
GRAD_CLIP = 1.0 #@param {type:'number'}
EARLY_STOP_PATIENCE = 3 #@param {type:'integer'}

assert TOKENIZER in {'word', 'char'}
assert DATASET_NAME in {'multi3_nlu_es_hotels', 'multi3_nlu_es_banking', 'multi3_nlu_es_all', 'toy_sintetico', 'simple_tico19_es', 'massive_es', 'mtop_es', 'tico19_es'}


## 3. Carga de corpus

Las funciones siguientes devuelven una lista de strings. Asi el resto del notebook no depende del formato original del dataset.


In [63]:
#@title Loaders de datasets espanoles
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

def run_cmd(cmd):
    print('+', ' '.join(cmd))
    subprocess.run(cmd, check=True)

def ensure_git_repo(url, path):
    path = Path(path)
    if not path.exists():
        run_cmd(['git', 'clone', '--depth', '1', url, str(path)])
    return path

def clean_text_list(texts):
    cleaned = []
    for text in texts:
        if not isinstance(text, str):
            continue
        text = re.sub(r'\s+', ' ', text.strip())
        if text:
            cleaned.append(text)
    return cleaned

def iter_json_records(obj):
    if isinstance(obj, list):
        for item in obj:
            yield from iter_json_records(item)
    elif isinstance(obj, dict):
        known = {'text', 'utterance', 'utt', 'sentence', 'original', 'simplification'}
        if known.intersection(obj.keys()):
            yield obj
        else:
            for value in obj.values():
                if isinstance(value, (dict, list)):
                    yield from iter_json_records(value)

def choose_text(record, preferred_keys):
    for key in preferred_keys:
        value = record.get(key)
        if isinstance(value, str) and value.strip():
            return value

    spanish_keys = ['es', 'es-ES', 'es_ES', 'spa', 'spa_Latn', 'spanish']
    for value in record.values():
        if isinstance(value, dict):
            for key in spanish_keys:
                nested = value.get(key)
                if isinstance(nested, str) and nested.strip():
                    return nested

    for value in record.values():
        if isinstance(value, str) and len(value.strip()) > 1:
            return value
    return None

def dataset_to_texts(ds, preferred_keys):
    texts = []
    if hasattr(ds, 'keys'):
        split_names = [s for s in ['train', 'validation', 'dev', 'test'] if s in ds]
        split_names = split_names or list(ds.keys())
        for split in split_names:
            for record in ds[split]:
                text = choose_text(record, preferred_keys)
                if text:
                    texts.append(text)
    else:
        for record in ds:
            text = choose_text(record, preferred_keys)
            if text:
                texts.append(text)
    return clean_text_list(texts)

def load_toy_sintetico(n=1500):
    alarm_times = ['a las 6', 'a las 7', 'a las 8', 'a las 9']
    days = ['hoy', 'manana', 'el lunes', 'el martes']
    alarm_forms = [
        'pon una alarma {day} {time}',
        'despiertame {day} {time}',
        'activa una alarma {day} {time}',
    ]
    cities = ['Monterrey', 'Madrid', 'Bogota', 'CDMX']
    weather_forms = [
        'que tiempo hara en {city}',
        'dime el clima en {city}',
        'como estara el tiempo en {city}',
    ]
    hotel_forms = [
        'quiero revisar mi reserva del hotel',
        'cancela mi reserva del hotel para manana',
        'necesito cambiar la fecha de llegada',
    ]
    rows = []
    for _ in range(n):
        family = random.choice(['alarm', 'weather', 'hotel'])
        if family == 'alarm':
            rows.append(random.choice(alarm_forms).format(day=random.choice(days), time=random.choice(alarm_times)))
        elif family == 'weather':
            rows.append(random.choice(weather_forms).format(city=random.choice(cities)))
        else:
            rows.append(random.choice(hotel_forms))
    return rows

def multi3_path_matches_domain(path, domain):
    if domain == 'all':
        return True
    parts = str(path).lower().replace('\\', '/').split('/')
    aliases = {
        'hotels': {'hotel', 'hotels'},
        'banking': {'bank', 'banking'},
    }.get(domain, {domain})
    return bool(aliases.intersection(parts))

def multi3_record_matches_domain(record, domain):
    if domain == 'all':
        return True
    uid = str(record.get('uid', '')).lower()
    if domain == 'hotels':
        return uid.startswith(('hotel_', 'hotels_'))
    if domain == 'banking':
        return uid.startswith(('bank_', 'banking_'))
    return False

def multi3_hf_json_paths(domain):
    from huggingface_hub import hf_hub_download, list_repo_files

    repo_id = 'uoe-nlp/multi3-nlu'
    files = list_repo_files(repo_id, repo_type='dataset')
    spanish_files = [p for p in files if p.startswith('spanish/') and p.endswith('.json')]
    domain_files = [p for p in spanish_files if multi3_path_matches_domain(p, domain)]
    selected_files = domain_files if domain_files else spanish_files
    paths = []
    for repo_file in selected_files:
        local_path = hf_hub_download(
            repo_id=repo_id,
            filename=repo_file,
            repo_type='dataset',
            local_dir=DATA_DIR / 'multi3-nlu-hf',
        )
        paths.append((Path(local_path), repo_file))
    return paths

def multi3_git_json_paths(domain):
    repo = ensure_git_repo('https://huggingface.co/datasets/uoe-nlp/multi3-nlu', DATA_DIR / 'multi3-nlu')
    all_paths = sorted(repo.glob('spanish/**/*.json'))
    domain_paths = [p for p in all_paths if multi3_path_matches_domain(p.relative_to(repo), domain)]
    selected_paths = domain_paths if domain_paths else all_paths
    return [(p, p.relative_to(repo).as_posix()) for p in selected_paths]

def load_multi3_nlu_es(domain='all'):
    texts = []
    try:
        json_paths = multi3_hf_json_paths(domain)
    except Exception as exc:
        print('No pude listar/descargar con huggingface_hub; intento git clone.', repr(exc))
        json_paths = multi3_git_json_paths(domain)

    if not json_paths:
        raise FileNotFoundError(f'No encontre archivos JSON espanoles para Multi3-NLU++ dominio={domain}')

    for fp, repo_file in json_paths:
        with fp.open(encoding='utf-8') as f:
            data = json.load(f)
        path_matches = multi3_path_matches_domain(repo_file, domain)
        for record in iter_json_records(data):
            if domain != 'all' and not (path_matches or multi3_record_matches_domain(record, domain)):
                continue
            text = choose_text(record, ['text', 'utterance', 'sentence'])
            if text:
                texts.append(text)

    texts = clean_text_list(texts)
    if not texts:
        raise ValueError(f'No encontre textos para Multi3-NLU++ dominio={domain}. Prueba multi3_nlu_es_all para diagnosticar.')
    return texts

def load_simple_tico19_es():
    repo = ensure_git_repo('https://github.com/MMU-TDMLab/SimpleTICO19.git', DATA_DIR / 'SimpleTICO19')
    texts = []
    csv_paths = sorted((repo / 'dataset').glob('simpletico19.*.es.csv'))
    for fp in csv_paths:
        with fp.open(encoding='utf-8', newline='') as f:
            reader = csv.DictReader(f)
            for row in reader:
                text = row.get(TEXT_FIELD_SIMPLE_TICO) or row.get('simplification') or row.get('original')
                if text:
                    texts.append(text)
    return clean_text_list(texts)

def load_massive_es():
    ds = load_dataset('AmazonScience/massive', 'es-ES')
    return dataset_to_texts(ds, ['utt', 'text', 'utterance'])

def load_mtop_es():
    last_error = None
    for args in [('tasksource/mtop', 'es'), ('tasksource/mtop',)]:
        try:
            ds = load_dataset(*args)
            texts = dataset_to_texts(ds, ['text', 'utterance', 'query', 'question'])
            if texts:
                return texts
        except Exception as exc:
            last_error = exc
    raise RuntimeError('No pude cargar MTOP desde Hugging Face') from last_error

def load_tico19_es():
    ds = load_dataset('SEACrowd/tico_19', trust_remote_code=True)
    return dataset_to_texts(ds, ['text', 'sentence', 'es', 'spa', 'spanish'])

LOADERS = {
    'toy_sintetico': load_toy_sintetico,
    'multi3_nlu_es_hotels': lambda: load_multi3_nlu_es('hotels'),
    'multi3_nlu_es_banking': lambda: load_multi3_nlu_es('banking'),
    'multi3_nlu_es_all': lambda: load_multi3_nlu_es('all'),
    'simple_tico19_es': load_simple_tico19_es,
    'massive_es': load_massive_es,
    'mtop_es': load_mtop_es,
    'tico19_es': load_tico19_es,
}

texts = LOADERS[DATASET_NAME]()
texts = clean_text_list(texts)
random.shuffle(texts)
if MAX_TEXTS and MAX_TEXTS > 0:
    texts = texts[:MAX_TEXTS]

if len(texts) < 3:
    raise ValueError(f'El corpus cargado es demasiado pequeno: {len(texts)} textos')

print(f'dataset: {DATASET_NAME}')
print(f'textos cargados: {len(texts):,}')
print('\nEjemplos:')
for sample_text in texts[:5]:
    print('-', sample_text[:220])


dataset: multi3_nlu_es_hotels
textos cargados: 1,009

Ejemplos:
- reserva una mesa para 5
- ¿a qué hora abre el gimnasio en la mañana?
- hasta el viernes
- ¿Por qué el servicio de limpieza no limpió mi habitación esta mañana?
- del 26 al 29 de mayo


## 4. Normalizacion, tokenizacion y vocabulario

Para corpus diminutos puedes usar `char`. Para Multi3-NLU++, Simple TICO-19 o MASSIVE, `word` suele ser suficiente para aprender la canalizacion completa antes de pasar a BPE.


In [64]:
#@title Tokenizacion y vocabulario
WORD_RE = re.compile(r"<[^>\s]+>|\w+|[^\w\s]", flags=re.UNICODE)

MONTHS_RE = r'enero|febrero|marzo|abril|mayo|junio|julio|agosto|septiembre|setiembre|octubre|noviembre|diciembre'
NUMBER_WORDS_RE = r'dos|tres|cuatro|cinco|seis|siete|ocho|nueve|diez|once|doce'

def normalize_slots(text):
    text = re.sub(r'\b\d{1,2}\s*[:.]\s*\d{2}\b', ' <hora> ', text)
    text = re.sub(r'\b\d{1,2}\s+y\s+(?:cuarto|media)\b', ' <hora> ', text, flags=re.IGNORECASE)
    text = re.sub(r'\b\d{1,2}\s+menos\s+\d{1,2}\b', ' <hora> ', text, flags=re.IGNORECASE)
    text = re.sub(r'\ba\s+l(?:a|as)\s+\d{1,2}\b', ' a las <hora> ', text, flags=re.IGNORECASE)
    text = re.sub(rf'\b({MONTHS_RE})\b', ' <mes> ', text, flags=re.IGNORECASE)
    text = re.sub(r'\b\d+\b', ' <num> ', text)
    text = re.sub(rf'\b({NUMBER_WORDS_RE})\b', ' <num> ', text, flags=re.IGNORECASE)
    return re.sub(r'\s+', ' ', text).strip()

def normalize(text):
    text = re.sub(r'\s+', ' ', text.strip())
    if LOWERCASE:
        text = text.lower()
    if NORMALIZE_SLOTS:
        text = normalize_slots(text)
    return text

def tokenize(text):
    text = normalize(text)
    if TOKENIZER == 'char':
        return list(text)
    return WORD_RE.findall(text)

split_at = max(1, int(0.9 * len(texts)))
train_texts = texts[:split_at]
val_texts = texts[split_at:] or texts[:max(1, min(100, len(texts)))]

counter = Counter()
for text in train_texts:
    counter.update(tokenize(text))

SPECIAL = ['<pad>', '<unk>', '<bos>', '<eos>']
itos = SPECIAL + [tok for tok, count in counter.most_common() if count >= MIN_FREQ and tok not in SPECIAL]
stoi = {tok: idx for idx, tok in enumerate(itos)}

PAD = stoi['<pad>']
UNK = stoi['<unk>']
BOS = stoi['<bos>']
EOS = stoi['<eos>']

def encode_texts(text_list):
    ids = []
    for text in text_list:
        ids.append(BOS)
        ids.extend(stoi.get(tok, UNK) for tok in tokenize(text))
        ids.append(EOS)
    return torch.tensor(ids, dtype=torch.long)

train_ids = encode_texts(train_texts)
val_ids = encode_texts(val_texts)
if len(val_ids) <= SEQ_LEN + 1:
    val_ids = train_ids.clone()

print(f'train_texts: {len(train_texts):,}')
print(f'val_texts: {len(val_texts):,}')
print(f'vocab_size: {len(itos):,}')
print(f'train_tokens: {len(train_ids):,}')
print(f'val_tokens: {len(val_ids):,}')


train_texts: 908
val_texts: 101
vocab_size: 1,028
train_tokens: 10,272
val_tokens: 1,155


## 5. Dataset de ventanas causales

Cada ejemplo usa `x = tokens[t:t+SEQ_LEN]` y `y = tokens[t+1:t+SEQ_LEN+1]`. Esa es la tarea clasica de predecir el siguiente token.


In [65]:
#@title Ventanas de entrenamiento
class WindowDataset(Dataset):
    def __init__(self, ids, seq_len=32, stride=1):
        if len(ids) <= seq_len + 1:
            raise ValueError('El corpus codificado es mas corto que SEQ_LEN. Reduce SEQ_LEN o carga mas texto.')
        self.ids = ids
        self.seq_len = seq_len
        self.stride = max(1, stride)
        self.n = 1 + (len(ids) - seq_len - 1) // self.stride

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        start = idx * self.stride
        x = self.ids[start : start + self.seq_len]
        y = self.ids[start + 1 : start + self.seq_len + 1]
        return x, y

train_ds = WindowDataset(train_ids, seq_len=SEQ_LEN, stride=STRIDE)
val_ds = WindowDataset(val_ids, seq_len=SEQ_LEN, stride=STRIDE)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

print(f'train_windows: {len(train_ds):,}')
print(f'val_windows: {len(val_ds):,}')


train_windows: 1,278
val_windows: 139


## 6. Modelo minimo: Embedding -> GRU -> Linear

Esta es la arquitectura recomendada en el reporte para empezar: pequena, facil de leer, y suficiente para aprender el flujo completo de un language model causal.


In [66]:
#@title Modelo GRU minimo
class TinyGRULM(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden_dim=256, num_layers=1):
        super().__init__()
        dropout = 0.1 if num_layers > 1 else 0.0
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        self.rnn = nn.GRU(
            emb_dim,
            hidden_dim,
            num_layers=num_layers,
            dropout=dropout,
            batch_first=True,
        )
        self.head = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, h=None):
        x = self.emb(x)
        out, h = self.rnn(x, h)
        logits = self.head(out)
        return logits, h

model = TinyGRULM(
    vocab_size=len(itos),
    emb_dim=EMB_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
).to(device)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)

n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'parametros: {n_params:,}')


TinyGRULM(
  (emb): Embedding(1028, 128, padding_idx=0)
  (rnn): GRU(128, 256, batch_first=True)
  (head): Linear(in_features=256, out_features=1028, bias=True)
)
parametros: 692,228


## 7. Entrenamiento y perplexity

La perplexity deberia bajar rapido en `toy_sintetico`. En corpus reales puede bajar mas lento, pero aun asi deberias ver una tendencia clara si el dataset cargo bien.


In [67]:
#@title Loop de entrenamiento
def safe_ppl(avg_loss):
    return float('inf') if avg_loss > 20 else math.exp(avg_loss)

def run_epoch(loader, train=True):
    model.train(mode=train)
    total_loss = 0.0
    total_tokens = 0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            logits, _ = model(x)
            loss = loss_fn(logits.reshape(-1, logits.size(-1)), y.reshape(-1))

            if train:
                opt.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                opt.step()

            n_tokens = (y != PAD).sum().item()
            total_loss += loss.item() * n_tokens
            total_tokens += n_tokens

    avg_loss = total_loss / max(1, total_tokens)
    return avg_loss, safe_ppl(avg_loss)

history = []
best_val_loss = float('inf')
best_epoch = 0
best_state = None
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_ppl = run_epoch(train_dl, train=True)
    val_loss, val_ppl = run_epoch(val_dl, train=False)
    improved = val_loss < best_val_loss

    if improved:
        best_val_loss = val_loss
        best_epoch = epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'train_ppl': train_ppl,
        'val_ppl': val_ppl,
        'is_best': improved,
    })
    print(
        f'epoch={epoch:02d} '
        f'train_loss={train_loss:.4f} train_ppl={train_ppl:.2f} '
        f'val_loss={val_loss:.4f} val_ppl={val_ppl:.2f} '
        f"{'*best*' if improved else ''}"
    )

    if EARLY_STOP_PATIENCE and epochs_without_improvement >= EARLY_STOP_PATIENCE:
        print(f'Early stopping: sin mejora en {EARLY_STOP_PATIENCE} epocas.')
        break

if best_state is not None:
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    print(f'Modelo restaurado desde epoch={best_epoch} con best_val_loss={best_val_loss:.4f}')


epoch=01 train_loss=5.6249 train_ppl=277.25 val_loss=4.5820 val_ppl=97.71 *best*
epoch=02 train_loss=4.0942 train_ppl=59.99 val_loss=4.0523 val_ppl=57.53 *best*
epoch=03 train_loss=3.5471 train_ppl=34.71 val_loss=3.7421 val_ppl=42.19 *best*
epoch=04 train_loss=3.1630 train_ppl=23.64 val_loss=3.5684 val_ppl=35.46 *best*
epoch=05 train_loss=2.8779 train_ppl=17.78 val_loss=3.4448 val_ppl=31.34 *best*
epoch=06 train_loss=2.6455 train_ppl=14.09 val_loss=3.3693 val_ppl=29.06 *best*
epoch=07 train_loss=2.4422 train_ppl=11.50 val_loss=3.3140 val_ppl=27.49 *best*
epoch=08 train_loss=2.2580 train_ppl=9.56 val_loss=3.2807 val_ppl=26.59 *best*
epoch=09 train_loss=2.0935 train_ppl=8.11 val_loss=3.2525 val_ppl=25.86 *best*
epoch=10 train_loss=1.9431 train_ppl=6.98 val_loss=3.2481 val_ppl=25.74 *best*
epoch=11 train_loss=1.8071 train_ppl=6.09 val_loss=3.2430 val_ppl=25.61 *best*
Modelo restaurado desde epoch=11 con best_val_loss=3.2430


## 8. Generacion

La generacion aqui usa muestreo multinomial con temperatura y `top_k`. Para corpus pequenos, prueba temperaturas entre `0.7` y `1.0`.


In [68]:
#@title Muestreo de texto
@torch.no_grad()
def sample(prompt='quiero revisar', max_new=18, temperature=0.55, top_k=6):
    model.eval()
    token_ids = [BOS] + [stoi.get(tok, UNK) for tok in tokenize(prompt)]

    for _ in range(max_new):
        context = torch.tensor([token_ids[-SEQ_LEN:]], dtype=torch.long, device=device)
        logits, _ = model(context)
        logits = logits[:, -1, :] / max(temperature, 1e-6)

        if top_k and top_k > 0 and top_k < logits.size(-1):
            values, _ = torch.topk(logits, top_k)
            cutoff = values[:, -1].unsqueeze(-1)
            logits = torch.where(logits < cutoff, torch.full_like(logits, -float('inf')), logits)

        probs = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1).item()
        if next_id == EOS:
            break
        token_ids.append(next_id)

    out_tokens = [itos[i] for i in token_ids[1:] if i not in {PAD, EOS}]
    if TOKENIZER == 'char':
        return ''.join(tok for tok in out_tokens if tok not in {'<bos>', '<unk>'})

    text = ' '.join(tok for tok in out_tokens if tok != '<bos>')
    text = re.sub(r'\s+([.,!?;:])', r'\1', text)
    return text

SAMPLE_MAX_NEW = 18
SAMPLE_TEMPERATURE = 0.55
SAMPLE_TOP_K = 6
SAMPLES_PER_PROMPT = 3
RENDER_SAMPLE_SLOTS = True

SLOT_RENDER_VALUES = {
    '<num>': ['1', '2', '3', '4', '5'],
    '<mes>': ['junio', 'julio', 'agosto', 'septiembre'],
    '<hora>': ['7:00', '8:30', '12:00', '18:00'],
}

def render_slots(text):
    def repl(match):
        token = match.group(0)
        return random.choice(SLOT_RENDER_VALUES.get(token, [token]))
    return re.sub(r'<num>|<mes>|<hora>', repl, text)

TOY_PROMPTS = ['quiero', 'pon una alarma', 'los sintomas', 'reserva del hotel']
HOTEL_PROMPTS = ['quiero reservar una habitación', 'necesito cambiar mi reserva', 'reserva del hotel por', 'quiero una habitación para']
DEFAULT_PROMPTS = ['quiero', 'necesito', 'dime', 'puedo']

if DATASET_NAME == 'toy_sintetico':
    prompts = TOY_PROMPTS
elif DATASET_NAME == 'multi3_nlu_es_hotels':
    prompts = HOTEL_PROMPTS
else:
    prompts = DEFAULT_PROMPTS

generated_samples = []
for prompt in prompts:
    for sample_idx in range(1, SAMPLES_PER_PROMPT + 1):
        generated = sample(prompt=prompt, max_new=SAMPLE_MAX_NEW, temperature=SAMPLE_TEMPERATURE, top_k=SAMPLE_TOP_K)
        rendered = render_slots(generated) if RENDER_SAMPLE_SLOTS else generated
        generated_samples.append({'prompt': prompt, 'sample_idx': sample_idx, 'text': generated, 'rendered_text': rendered})
        print(f'PROMPT: {prompt} [{sample_idx}]')
        print(generated)
        if RENDER_SAMPLE_SLOTS and rendered != generated:
            print('rendered:', rendered)
        print()


PROMPT: quiero reservar una habitación [1]
quiero reservar una habitación para <num> adultos y un niño
rendered: quiero reservar una habitación para 3 adultos y un niño

PROMPT: quiero reservar una habitación [2]
quiero reservar una habitación desde el <num> de <mes>
rendered: quiero reservar una habitación desde el 5 de julio

PROMPT: quiero reservar una habitación [3]
quiero reservar una habitación para <num> adultos
rendered: quiero reservar una habitación para 4 adultos

PROMPT: necesito cambiar mi reserva [1]
necesito cambiar mi reserva de habitación para el <num> de <mes>
rendered: necesito cambiar mi reserva de habitación para el 1 de julio

PROMPT: necesito cambiar mi reserva [2]
necesito cambiar mi reserva para el <num> de <mes>
rendered: necesito cambiar mi reserva para el 2 de agosto

PROMPT: necesito cambiar mi reserva [3]
necesito cambiar mi reserva de hotel

PROMPT: reserva del hotel por [1]
reserva del hotel por <num> noches desde el <num> de <mes>
rendered: reserva del 

## 9. Guardar checkpoint

El checkpoint guarda pesos, vocabulario y configuracion basica. En Colab puedes descargar el archivo desde el panel lateral de archivos.


In [69]:
#@title Guardar modelo
SAVE_PATH = 'tiny_gru_lm_es.pt'
checkpoint = {
    'model_state_dict': model.state_dict(),
    'itos': itos,
    'stoi': stoi,
    'config': {
        'dataset_name': DATASET_NAME,
        'tokenizer': TOKENIZER,
        'lowercase': LOWERCASE,
        'normalize_slots': NORMALIZE_SLOTS,
        'seq_len': SEQ_LEN,
        'stride': STRIDE,
        'lr': LR,
        'weight_decay': WEIGHT_DECAY,
        'emb_dim': EMB_DIM,
        'hidden_dim': HIDDEN_DIM,
        'num_layers': NUM_LAYERS,
    },
    'best_epoch': best_epoch,
    'best_val_loss': best_val_loss,
    'history': history,
}
torch.save(checkpoint, SAVE_PATH)
print(f'checkpoint guardado en: {SAVE_PATH}')


checkpoint guardado en: tiny_gru_lm_es.pt


## 10. Exportar resultados

Esta celda guarda una carpeta `SimpleNeuralLM/runs/latest/` cuando detecta el repo, o `runs/latest/` si el notebook corre aislado en Colab. Tambien crea `latest.zip` para descargar/sincronizar los resultados con el workspace local.


In [70]:
#@title Exportar resultados para iteracion
import shutil

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != 'SimpleNeuralLM' and (PROJECT_DIR / 'SimpleNeuralLM').exists():
    PROJECT_DIR = PROJECT_DIR / 'SimpleNeuralLM'

RUN_DIR = PROJECT_DIR / 'runs' / 'latest'
RUN_DIR.mkdir(parents=True, exist_ok=True)

best_row = min(history, key=lambda row: row['val_loss']) if history else {}
best_val_ppl = best_row.get('val_ppl')

run_config = {
    'dataset_name': DATASET_NAME,
    'tokenizer': TOKENIZER,
    'lowercase': LOWERCASE,
    'normalize_slots': NORMALIZE_SLOTS,
    'max_texts': MAX_TEXTS,
    'min_freq': MIN_FREQ,
    'seq_len': SEQ_LEN,
    'stride': STRIDE,
    'batch_size': BATCH_SIZE,
    'epochs_requested': EPOCHS,
    'emb_dim': EMB_DIM,
    'hidden_dim': HIDDEN_DIM,
    'num_layers': NUM_LAYERS,
    'lr': LR,
    'weight_decay': WEIGHT_DECAY,
    'grad_clip': GRAD_CLIP,
    'early_stop_patience': EARLY_STOP_PATIENCE,
    'sample_max_new': globals().get('SAMPLE_MAX_NEW'),
    'sample_temperature': globals().get('SAMPLE_TEMPERATURE'),
    'sample_top_k': globals().get('SAMPLE_TOP_K'),
    'samples_per_prompt': globals().get('SAMPLES_PER_PROMPT'),
    'render_sample_slots': globals().get('RENDER_SAMPLE_SLOTS'),
    'device': device,
    'cwd': str(Path.cwd()),
    'project_dir': str(PROJECT_DIR),
}

summary = {
    'dataset_name': DATASET_NAME,
    'num_texts': len(texts),
    'train_texts': len(train_texts),
    'val_texts': len(val_texts),
    'vocab_size': len(itos),
    'train_tokens': int(len(train_ids)),
    'val_tokens': int(len(val_ids)),
    'train_windows': len(train_ds),
    'val_windows': len(val_ds),
    'best_epoch': int(best_epoch),
    'best_val_loss': float(best_val_loss),
    'best_val_ppl': float(best_val_ppl) if best_val_ppl is not None else None,
    'final_epoch': int(history[-1]['epoch']) if history else None,
    'checkpoint_path': SAVE_PATH,
    'run_dir': str(RUN_DIR),
}

with (RUN_DIR / 'config.json').open('w', encoding='utf-8') as f:
    json.dump(run_config, f, ensure_ascii=False, indent=2)

with (RUN_DIR / 'summary.json').open('w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

metric_fields = ['epoch', 'train_loss', 'val_loss', 'train_ppl', 'val_ppl', 'is_best']
with (RUN_DIR / 'metrics.csv').open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=metric_fields)
    writer.writeheader()
    for row in history:
        writer.writerow({field: row.get(field) for field in metric_fields})

samples = globals().get('generated_samples', [])
if not samples:
    samples = []
    for prompt in prompts:
        for sample_idx in range(1, globals().get('SAMPLES_PER_PROMPT', 1) + 1):
            samples.append({
                'prompt': prompt,
                'sample_idx': sample_idx,
                'text': sample(
                    prompt=prompt,
                    max_new=globals().get('SAMPLE_MAX_NEW', 18),
                    temperature=globals().get('SAMPLE_TEMPERATURE', 0.55),
                    top_k=globals().get('SAMPLE_TOP_K', 6),
                ),
            })
    if globals().get('RENDER_SAMPLE_SLOTS'):
        for item in samples:
            item['rendered_text'] = render_slots(item['text'])

OFF_DOMAIN_TERMS = ['mesa', 'cena', 'restaurante', 'spa', 'cepillo']

def sample_flags(item):
    prompt = item.get('prompt', '').lower()
    text = item.get('text', '').lower()
    flags = []
    if '<unk>' in text:
        flags.append('unk_token')
    if any(term in text for term in OFF_DOMAIN_TERMS):
        flags.append('possible_domain_drift')
    if len(text.split()) <= len(prompt.split()) + 1:
        flags.append('short_or_echo')
    if re.search(r'\b(\w+)\s+\1\b', text):
        flags.append('adjacent_repeat')
    if 'habitación para la habitación' in text or 'habitaciÃ³n para la habitaciÃ³n' in text:
        flags.append('phrase_repeat')
    return flags

for item in samples:
    item['flags'] = sample_flags(item)

quality_summary = {
    'num_samples': len(samples),
    'flagged_samples': sum(1 for item in samples if item.get('flags')),
    'flag_counts': dict(Counter(flag for item in samples for flag in item.get('flags', []))),
}
summary['sample_quality'] = quality_summary

best_is_final = bool(history) and best_epoch == history[-1]['epoch']
if quality_summary['flagged_samples'] == 0 and best_is_final:
    agent_recommendation = 'stable_keep_current_settings'
elif quality_summary['flagged_samples'] > 0:
    agent_recommendation = 'review_sampling_or_prompts'
elif not best_is_final:
    agent_recommendation = 'reduce_epochs_or_use_best_checkpoint'
else:
    agent_recommendation = 'inspect_manually'

summary['agent_recommendation'] = agent_recommendation
run_config['off_domain_terms'] = OFF_DOMAIN_TERMS

with (RUN_DIR / 'summary.json').open('w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

with (RUN_DIR / 'config.json').open('w', encoding='utf-8') as f:
    json.dump(run_config, f, ensure_ascii=False, indent=2)

with (RUN_DIR / 'samples.md').open('w', encoding='utf-8') as f:
    f.write(f'# Samples: {DATASET_NAME}\n\n')
    for item in samples:
        sample_idx = item.get('sample_idx')
        suffix = f' [{sample_idx}]' if sample_idx is not None else ''
        f.write(f"## Prompt: {item['prompt']}{suffix}\n")
        f.write(item['text'].strip() + '\n')
        rendered = item.get('rendered_text')
        if rendered and rendered != item['text']:
            f.write(f"Rendered: {rendered.strip()}\n")
        if item.get('flags'):
            f.write(f"Flags: {', '.join(item['flags'])}\n")
        f.write('\n')

agent_artifact = {
    'config': run_config,
    'summary': summary,
    'metrics': history,
    'samples': samples,
}

with (RUN_DIR / 'agent_artifact.json').open('w', encoding='utf-8') as f:
    json.dump(agent_artifact, f, ensure_ascii=False, indent=2)

checkpoint_file = Path(SAVE_PATH)
if checkpoint_file.exists():
    shutil.copyfile(checkpoint_file, RUN_DIR / checkpoint_file.name)

zip_path = shutil.make_archive(str(RUN_DIR.parent / 'latest'), 'zip', root_dir=RUN_DIR.parent, base_dir='latest')

print(f'Resultados exportados en: {RUN_DIR}')
print(f'ZIP exportado en: {zip_path}')
print(f'Recomendacion agente: {agent_recommendation}')
print('Archivos:', ', '.join(p.name for p in sorted(RUN_DIR.iterdir())))
print('AGENT_ARTIFACT_JSON_START')
print(json.dumps(agent_artifact, ensure_ascii=False))
print('AGENT_ARTIFACT_JSON_END')


Resultados exportados en: /content/runs/latest
ZIP exportado en: /content/runs/latest.zip
Recomendacion agente: stable_keep_current_settings
Archivos: agent_artifact.json, config.json, metrics.csv, samples.md, summary.json, tiny_gru_lm_es.pt
AGENT_ARTIFACT_JSON_START
{"config": {"dataset_name": "multi3_nlu_es_hotels", "tokenizer": "word", "lowercase": true, "normalize_slots": true, "max_texts": 20000, "min_freq": 1, "seq_len": 48, "stride": 8, "batch_size": 64, "epochs_requested": 11, "emb_dim": 128, "hidden_dim": 256, "num_layers": 1, "lr": 0.001, "weight_decay": 0.0001, "grad_clip": 1.0, "early_stop_patience": 3, "sample_max_new": 18, "sample_temperature": 0.55, "sample_top_k": 6, "samples_per_prompt": 3, "render_sample_slots": true, "device": "cpu", "cwd": "/content", "project_dir": "/content", "off_domain_terms": ["mesa", "cena", "restaurante", "spa", "cepillo"]}, "summary": {"dataset_name": "multi3_nlu_es_hotels", "num_texts": 1009, "train_texts": 908, "val_texts": 101, "vocab_siz

## Siguientes experimentos utiles

- Usa `multi3_nlu_es_hotels` para entrenar solo con hoteles, o `multi3_nlu_es_banking` para banca.
- Prueba `TOKENIZER = 'char'` si `SEQ_LEN` pequeno o vocabulario grande te da problemas.
- Sube `EPOCHS` a 10-20 si la perplexity de validacion sigue bajando.
- Cuando esta version funcione, el siguiente paso natural es reemplazar `word/char` por BPE con `sentencepiece`.
